# ***SCRAPING DATA***

In [1]:
!pip install google-play-scraper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.2 MB/s eta 0:00:00


In [7]:
from google_play_scraper import reviews, Sort
from datetime import datetime
import csv
import hashlib
import difflib

app_id = 'app.bpjs.mobile'

def get_reviews_by_date(app_id, lang='id', country='id', count=10000, start_date='2025-01-01', end_date='2026-03-04'):
    all_reviews = []
    continuation_token = None

    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt = datetime.strptime(end_date, "%Y-%m-%d")

    while True:
        try:
            result, continuation_token = reviews(
                app_id,
                lang=lang,
                country=country,
                sort=Sort.NEWEST,
                count=count,
                continuation_token=continuation_token
            )

            if not result:
                break

            for r in result:
                review_date = r.get('at')
                if review_date is None:
                    continue

                if start_dt <= review_date <= end_dt:
                    rec = {
                        'reviewId': r.get('reviewId'),
                        'userName': r.get('userName'),
                        'userImage': r.get('userImage'),
                        'score': r.get('score'),
                        'at': review_date.isoformat(),
                        'content': r.get('content'),
                        'thumbsUpCount': r.get('thumbsUp'),
                        'replyContent': r.get('replyContent'),
                        'replyAt': r.get('replyAt').isoformat() if r.get('replyAt') else None
                    }
                    all_reviews.append(rec)

            if continuation_token is None:
                break

        except Exception as e:
            print("Error:", e)
            break

    return all_reviews

def fingerprint_text(text, n_chars=80):
    if not text:
        return None
    t = text.strip().lower()[:n_chars]
    return hashlib.sha1(t.encode('utf-8')).hexdigest()

def fingerprint_image(url):
    if not url:
        return None
    return hashlib.sha1(url.encode('utf-8')).hexdigest()

def assign_probable_user_id(reviews):
    for r in reviews:
        uname = (r.get('userName') or '').strip()
        if uname and uname.lower() != 'pengguna google':
            # normalize small whitespace/case differences
            pid = 'name:' + '_'.join(uname.split()).lower()
        else:
            img_fp = fingerprint_image(r.get('userImage'))
            if img_fp:
                pid = 'img:' + img_fp
            else:
                txt_fp = fingerprint_text(r.get('content') or '')
                if txt_fp:
                    pid = 'txt:' + txt_fp
                else:
                    # fallback to reviewId (unique per review) so it won't group
                    pid = 'rid:' + (r.get('reviewId') or 'unknown')
        r['probable_user_id'] = pid

    TEXT_SIM_THRESHOLD = 0.92
    text_groups = {}
    for r in reviews:
        if r['probable_user_id'].startswith('txt:'):
            base_txt = (r.get('content') or '').strip()
            merged = False
            for key, rep in text_groups.items():
                ratio = difflib.SequenceMatcher(None, base_txt, rep['sample']).ratio()
                if ratio >= TEXT_SIM_THRESHOLD:
                    r['probable_user_id'] = key
                    merged = True
                    break
            if not merged:
                key = r['probable_user_id']
                text_groups[key] = {'sample': base_txt}
    return reviews

def save_reviews_to_csv(reviews, filename='hasil_scraper_ulasan_app_mobile jkn.csv'):
    headers = ['probable_user_id','reviewId','userName','userImage','score','at','content','thumbsUpCount','replyContent','replyAt']
    with open(filename, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        for r in reviews:
            writer.writerow({k: r.get(k) for k in headers})
    print(f"Saved {len(reviews)} rows to {filename}")

filtered_reviews = get_reviews_by_date(
    app_id,
    lang='id',
    country='id',
    count=100000,
    start_date='2025-01-01',
    end_date='2026-03-04'
)

print("Total reviews fetched:", len(filtered_reviews))

if filtered_reviews:
    filtered_reviews = assign_probable_user_id(filtered_reviews)
    for i, r in enumerate(filtered_reviews[:10], 1):
        print(f"\n[{i}] probable_user_id: {r['probable_user_id']}")
        print(" userName:", r['userName'])
        print(" userImage:", r['userImage'])
        print(" date:", r['at'])
        print(" score:", r['score'])
        print(" content:", (r['content'] or "")[:200])

    save_reviews_to_csv(filtered_reviews)

Total reviews fetched: 68575

[1] probable_user_id: img:c72c5fd7c20d0497d4a734ac427fe3d4a5b4e911
 userName: Pengguna Google
 userImage: https://play-lh.googleusercontent.com/EGemoI2NTXmTsBVtJqk8jxF9rh8ApRWfsIMQSt2uE4OcpQqbFu7f7NbTK05lx80nuSijCz7sc3a277R67g
 date: 2026-03-03T23:57:07
 score: 1
 content: disuruh verifikasi wajah, tapi selalu gagal, padahal sy sudah memenuhi persyaratannya, saya mau verifikasi melalui email, gak bisa disuruh melalui nomor hp, tapi nomor hp saya yang terdaftar sudah tid

[2] probable_user_id: img:c72c5fd7c20d0497d4a734ac427fe3d4a5b4e911
 userName: Pengguna Google
 userImage: https://play-lh.googleusercontent.com/EGemoI2NTXmTsBVtJqk8jxF9rh8ApRWfsIMQSt2uE4OcpQqbFu7f7NbTK05lx80nuSijCz7sc3a277R67g
 date: 2026-03-03T23:21:07
 score: 5
 content: bagus

[3] probable_user_id: img:c72c5fd7c20d0497d4a734ac427fe3d4a5b4e911
 userName: Pengguna Google
 userImage: https://play-lh.googleusercontent.com/EGemoI2NTXmTsBVtJqk8jxF9rh8ApRWfsIMQSt2uE4OcpQqbFu7f7NbTK05lx80nuSijC